# WAAM Twin Cloud Smoke Test

This notebook is designed for **Google Colab** and similar cloud notebook environments.

It focuses on **headless bring-up** of `waam_twin`:

- install dependencies
- clone or use an uploaded repo
- run a short bead-on-plate smoke test
- inspect telemetry
- export VTK / research bundle
- preview a few scalar fields in Python

Use this notebook for environment verification only.

For real cloud-side job tuning and production-style runs, use:

- `waam_twin/notebooks/cloud_production_workflow.ipynb`

Notes:

- The desktop **Taichi GGUI viewer** is not the main target here.
- Prefer **CPU** first for portability; switch to CUDA only after the smoke test works.
- This notebook supports both the direct repo layout and the older nested local-dev layout.

## 1. Get the repo into the notebook runtime

Choose one:

1. Upload a zip of the direct repo and unzip it.
2. Clone your repo from GitHub.
3. Mount Google Drive and point to an existing checkout.

This notebook supports both the direct repo and the older nested local-dev layout.

In [ ]:
# Optional: clone the direct repo into the runtime.

REPO_URL = "https://github.com/THEDARKDEACON/waam_solver.git"

from pathlib import Path
import os
import subprocess

clone_dir = Path("/content/waam_twin")
if REPO_URL and not clone_dir.exists():
    subprocess.run(["git", "clone", REPO_URL, str(clone_dir)], check=True)

candidates = [
    clone_dir,
    Path("/content/waam_solver"),
    Path("/content/FYP22-01/waam_twin"),
    Path("/content/FYP22-01"),
]
root = next((p for p in candidates if p.exists() and (p / "requirements.txt").exists()), None)
if root is None:
    raise FileNotFoundError("Could not locate repo root.")

print("Repo root:", root)
print("Exists:", root.exists())
print("Repo URL:", REPO_URL)

In [ ]:
# Install dependencies.
# In Colab, %pip is preferred so the notebook kernel sees new packages.

from pathlib import Path

req = root / "requirements.txt"
assert req.exists(), f"Missing requirements file: {req}"

%pip install -r {req}

In [ ]:
# Put the repo parent on sys.path and choose a backend.

import os
import sys
from pathlib import Path

parent = root.parent
if str(parent) not in sys.path:
    sys.path.insert(0, str(parent))

# Portable default.
os.environ["WAAM_BACKEND"] = os.environ.get("WAAM_BACKEND", "cpu")
print("WAAM_BACKEND =", os.environ["WAAM_BACKEND"])

# Optional if you want to try GPU later:
# os.environ["WAAM_BACKEND"] = "cuda"

## 2. Run a short bead-on-plate smoke test

This uses the packaged job YAML and a small number of steps so you can check whether the simulator boots and produces sensible telemetry.

In [ ]:
from waam_twin.platform import init_taichi
from waam_twin import WAAMTwin

init_taichi(backend=os.environ["WAAM_BACKEND"])

job_path = root / "jobs" / "examples" / "bead_on_plate.yaml"
twin = WAAMTwin.from_job(job_path)
twin.reset()

# Short smoke run. Increase to 1000-3000 for a more developed pool.
twin.run_path(job_path, n_steps=400)
telem = twin.get_telemetry()
telem

## 3. Export a research bundle

This writes volume/surface/tracer VTK plus JSON sidecars. Those outputs are usually more useful in cloud notebooks than the desktop viewer.

In [ ]:
from pathlib import Path

out_dir = root / "viewer_output" / "cloud_bundle"
paths = twin.export_research_bundle(out_dir, tag="colab_smoke", tiers=(0, 1, 2, 3))
paths

## 4. Inspect the exported `.vti` in Python

This is a lightweight alternative to ParaView when you only want quick range checks and a few slice plots.

In [ ]:
import json
import numpy as np
import pyvista as pv

vti_path = Path(paths["volume"])
grid = pv.read(str(vti_path))

print("VTI:", vti_path)
print("n_cells:", grid.n_cells)
print("spacing:", grid.spacing)
print("arrays:", sorted(grid.cell_data.keys()))

for name in ["Temperature_K", "Liquid_Fraction", "Speed_ms", "Curvature_kappa"]:
    if name in grid.cell_data:
        arr = np.asarray(grid.cell_data[name])
        print(name, "min=", np.nanmin(arr), "max=", np.nanmax(arr), "mean=", np.nanmean(arr))

meta_path = Path(paths["meta"])
meta = json.loads(meta_path.read_text())
meta["telemetry"]

In [ ]:
# Quick slice preview for a scalar field.

import matplotlib.pyplot as plt
import numpy as np

field = "Temperature_K"  # try Liquid_Fraction, Speed_ms, T_max_K
dims = tuple(int(v - 1) for v in grid.dimensions)
arr = np.asarray(grid.cell_data[field]).reshape(dims, order="F")
mid_y = arr.shape[1] // 2
sl = arr[:, mid_y, :].T

plt.figure(figsize=(10, 4))
plt.imshow(sl, origin="lower", aspect="auto", cmap="inferno")
plt.colorbar(label=field)
plt.title(f"Mid-Y slice: {field}")
plt.xlabel("x cell")
plt.ylabel("z cell")
plt.show()

## 5. Suggested cloud workflow

For Colab / cloud vendors, I would go about it this way:

1. **Start on CPU** with a short smoke test.
2. **Use `WAAMTwin.from_job(...)`** so the same YAML jobs work locally and in cloud.
3. **Export VTK bundles** instead of relying on the desktop viewer.
4. **Inspect telemetry and scalar ranges** in the notebook first.
5. Only then scale up:
   - more steps
   - `standard` / `high` presets
   - CUDA backend if the runtime supports it

A practical split is:

- **Colab / notebooks**: smoke tests, telemetry, VTK generation, 2D slices
- **Local desktop / ParaView**: richer interactive inspection of `.vti` / `.vtp`

If you want, the next step can be a second notebook for:

- parameter sweeps (`travel_speed_mm_s`, `transfer_mode`, `wire_feed_m_min`)
- batch export of multiple runs
- automatic comparison plots of `bead_height_mm`, `pool_depth_mm`, and `mass_balance_ratio`